# Following an Answer Through a Transformer

We will trace how `gpt2-small` turns the prompt `The capital of France is Paris. The capital of Germany is` into the next-token prediction ` Berlin`.

## Research question
When does the answer become visible, which components push it upward, and which of those components actually matter causally?

## Investigation plan
1. Watch the residual stream evolve checkpoint by checkpoint.
2. Track competing tokens across pre-attention, post-attention, and post-MLP states.
3. Use direct logit attribution to form a mechanistic hypothesis.
4. Inspect the attention patterns of the strongest candidate heads.
5. Test the story with activation patching and head ablation.
6. Close with why this kind of analysis matters in practice.

## Why this matters in practice
Mechanistic analysis is not just curiosity work. If we can localize where an answer forms, we get better tools for debugging regressions, explaining model behavior to non-specialists, stress-testing deployment changes, and reasoning about trust when models are used in real workflows.


## Environment and scope

We use `gpt2-small` on purpose. It is small enough to inspect end to end, but still rich enough to show residual-stream accumulation, competing candidates, direct logit attribution, attention patterns, activation patching, and head ablation in one notebook.

This notebook should run on CPU, although the causal sections are faster on GPU. The focus here is not benchmark performance. It is building a clean, inspectable story about how one answer forms inside the model.


In [ ]:
import importlib.metadata as md
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from matplotlib.colors import TwoSlopeNorm

from transformer_lens import HookedTransformer, patching
from transformer_lens.utils import get_act_name

warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:0.4f}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "gpt2-small"

PROMPT = "The capital of France is Paris. The capital of Germany is"
TARGET_ANSWER = " Berlin"
RIVAL_ANSWER = " Paris"

CLEAN_PROMPT = PROMPT
CORRUPTED_PROMPT = "The capital of France is Paris. The capital of France is"

TOP_K = 8
TOP_COMPONENTS = 12
HEADS_TO_VISUALIZE = 3
HEADS_TO_ABLATE = 6

COLORS = {
    "target": "#0F4C81",
    "rival": "#B55239",
    "accent": "#2F855A",
    "neutral": "#4A5568",
}

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.18,
        "grid.linestyle": "--",
        "axes.titleweight": "semibold",
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.frameon": False,
    }
)

print("Using device:", DEVICE)
print("torch:", torch.__version__)
print("transformer-lens:", md.version("transformer-lens"))


def display_takeaways(title, what, change, why, practice=None):
    lines = [
        f"### {title}",
        f"- What we are looking at: {what}",
        f"- What changes here: {change}",
        f"- Why it matters: {why}",
    ]
    if practice:
        lines.append(f"- Why this matters in practice: {practice}")
    display(Markdown("\n".join(lines)))


def visible_token(token_str):
    return token_str.replace(" ", "[space]").replace("\n", "\\n")


def safe_to_single_token(model, token_str):
    token_ids = model.to_tokens(token_str, prepend_bos=False).squeeze(0).tolist()
    if len(token_ids) != 1:
        raise ValueError(
            f"Expected a single token for {token_str!r}, but got {len(token_ids)} tokens: {token_ids}"
        )
    return token_ids[0]


def token_id_to_str(model, token_id):
    return model.to_string([token_id]).replace("\n", "\\n")


def final_token_logits(logits):
    if logits.ndim == 3:
        return logits[:, -1, :]
    if logits.ndim == 2:
        return logits[-1, :].unsqueeze(0)
    raise ValueError(f"Unexpected logits shape: {tuple(logits.shape)}")


def get_token_rank(logits_1d, token_id):
    sorted_ids = torch.argsort(logits_1d, descending=True)
    rank = (sorted_ids == token_id).nonzero(as_tuple=False)
    return int(rank[0].item()) + 1


def make_metric_for_answer(answer_token, rival_token=None):
    def metric(logits):
        last = final_token_logits(logits)
        answer_logit = last[:, answer_token]
        if rival_token is None:
            return answer_logit.mean()
        rival_logit = last[:, rival_token]
        return (answer_logit - rival_logit).mean()

    return metric


def top_predictions_table(model, logits_1d, k=8):
    probs = F.softmax(logits_1d, dim=-1)
    values, token_ids = torch.topk(probs, k=k)
    rows = []
    for rank, (probability, token_id) in enumerate(zip(values.tolist(), token_ids.tolist()), start=1):
        rows.append(
            {
                "rank": rank,
                "token": visible_token(token_id_to_str(model, token_id)),
                "token_id": token_id,
                "probability": probability,
            }
        )
    return pd.DataFrame(rows)


def parse_checkpoint_label(label):
    label = str(label)
    if label in {"embed", "pos_embed"}:
        return {"layer": -1, "phase": label, "label_short": label.replace("_", " ")}
    if "_" not in label:
        return {"layer": None, "phase": label, "label_short": label}

    layer_text, phase = label.split("_", 1)
    short_phase = {"pre": "pre", "mid": "attn", "post": "mlp"}.get(phase, phase)
    long_phase = {"pre": "pre-attn", "mid": "post-attn", "post": "post-mlp"}.get(phase, phase)
    try:
        layer_value = int(layer_text)
        label_short = f"L{layer_value} {short_phase}"
    except ValueError:
        layer_value = None
        label_short = label
    return {"layer": layer_value, "phase": long_phase, "label_short": label_short}


def build_checkpoint_dataframe(model, checkpoint_labels, checkpoint_logits_last_pos, resid_last_pos, target_token, rival_token):
    probs = F.softmax(checkpoint_logits_last_pos, dim=-1)
    rows = []
    for checkpoint_idx, label in enumerate(checkpoint_labels):
        meta = parse_checkpoint_label(label)
        top_id = int(probs[checkpoint_idx].argmax().item())
        rows.append(
            {
                "checkpoint_idx": checkpoint_idx,
                "label": str(label),
                "label_short": meta["label_short"],
                "layer": meta["layer"],
                "phase": meta["phase"],
                "target_token": visible_token(token_id_to_str(model, target_token)),
                "rival_token": visible_token(token_id_to_str(model, rival_token)),
                "target_prob": probs[checkpoint_idx, target_token].item(),
                "rival_prob": probs[checkpoint_idx, rival_token].item(),
                "logit_diff": (
                    checkpoint_logits_last_pos[checkpoint_idx, target_token]
                    - checkpoint_logits_last_pos[checkpoint_idx, rival_token]
                ).item(),
                "target_rank": get_token_rank(checkpoint_logits_last_pos[checkpoint_idx], target_token),
                "rival_rank": get_token_rank(checkpoint_logits_last_pos[checkpoint_idx], rival_token),
                "top_prediction": visible_token(token_id_to_str(model, top_id)),
                "residual_norm": resid_last_pos[checkpoint_idx].norm().item(),
            }
        )
    return pd.DataFrame(rows)


In [ ]:
def choose_candidate_token_ids(checkpoint_logits_last_pos, target_token, rival_token, max_tokens=7):
    probs = F.softmax(checkpoint_logits_last_pos, dim=-1)
    anchor_points = sorted({0, len(checkpoint_logits_last_pos) // 4, len(checkpoint_logits_last_pos) // 2, (3 * len(checkpoint_logits_last_pos)) // 4, len(checkpoint_logits_last_pos) - 1})
    candidate_ids = {target_token, rival_token}
    for anchor in anchor_points:
        candidate_ids.update(torch.topk(probs[anchor], k=4).indices.tolist())
    other_ids = sorted(
        [token_id for token_id in candidate_ids if token_id not in {target_token, rival_token}],
        key=lambda token_id: probs[:, token_id].max().item(),
        reverse=True,
    )
    return ([target_token, rival_token] + other_ids)[:max_tokens]


def build_candidate_matrix(model, checkpoint_logits_last_pos, candidate_token_ids):
    probs = F.softmax(checkpoint_logits_last_pos, dim=-1).detach().cpu().numpy()
    token_labels = [visible_token(token_id_to_str(model, token_id)) for token_id in candidate_token_ids]
    matrix = np.vstack([probs[:, token_id] for token_id in candidate_token_ids])
    return token_labels, matrix


def extract_last_position_scores(tensor):
    scores = tensor.detach().cpu().squeeze()
    if scores.ndim == 1:
        return scores
    scores = scores[..., -1]
    if scores.ndim > 1:
        scores = scores.reshape(scores.shape[0], -1).sum(dim=-1)
    return scores


def attrs_to_df(labels, attrs):
    scores = extract_last_position_scores(attrs).numpy()
    return pd.DataFrame({"component": list(labels), "logit_attr": scores}).sort_values("logit_attr", ascending=False).reset_index(drop=True)


def head_attrs_to_df(model, attrs):
    scores = extract_last_position_scores(attrs).numpy().reshape(model.cfg.n_layers, model.cfg.n_heads)
    rows = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            rows.append({"layer": layer, "head": head, "label": f"L{layer}H{head}", "logit_attr": scores[layer, head]})
    return pd.DataFrame(rows).sort_values("logit_attr", ascending=False).reset_index(drop=True)


def normalize_patch_scores(scores, clean_score, corrupted_score):
    denominator = clean_score - corrupted_score
    if abs(denominator) < 1e-6:
        return scores * 0.0
    return (scores - corrupted_score) / denominator


def head_scores_to_df(head_scores):
    rows = []
    for layer in range(head_scores.shape[0]):
        for head in range(head_scores.shape[1]):
            rows.append({"layer": layer, "head": head, "label": f"L{layer}H{head}", "score": head_scores[layer, head].item()})
    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)


def zero_head_hook(head_idx):
    def hook_fn(z, hook):
        z = z.clone()
        z[:, :, head_idx, :] = 0.0
        return z

    return hook_fn


def ablate_single_head_and_score(model, tokens, layer, head, metric_fn, baseline_score):
    with torch.inference_mode():
        ablated_logits = model.run_with_hooks(tokens, fwd_hooks=[(get_act_name("z", layer), zero_head_hook(head))])
    ablated_score = metric_fn(ablated_logits).item()
    return {"layer": layer, "head": head, "label": f"L{layer}H{head}", "baseline_score": baseline_score, "ablated_score": ablated_score, "delta": ablated_score - baseline_score}


def plot_emergence_overview(checkpoint_df):
    x = checkpoint_df["checkpoint_idx"]
    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, constrained_layout=True)
    axes[0].plot(x, checkpoint_df["target_prob"], marker="o", color=COLORS["target"], linewidth=2, label=checkpoint_df["target_token"].iloc[0])
    axes[0].plot(x, checkpoint_df["rival_prob"], marker="o", color=COLORS["rival"], linewidth=2, label=checkpoint_df["rival_token"].iloc[0])
    axes[0].set_title("When does the answer overtake the rival?")
    axes[0].set_ylabel("Probability at final position")
    axes[0].legend(ncol=2)
    axes[1].plot(x, checkpoint_df["logit_diff"], marker="o", color=COLORS["accent"], linewidth=2)
    axes[1].axhline(0, color=COLORS["neutral"], linestyle="--", linewidth=1)
    axes[1].set_ylabel("Target - rival logit")
    axes[1].set_xlabel("Residual checkpoint")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(checkpoint_df["label_short"], rotation=90)
    return fig


def plot_candidate_heatmap(token_labels, matrix, checkpoint_labels):
    fig, ax = plt.subplots(figsize=(16, 5), constrained_layout=True)
    image = ax.imshow(matrix, aspect="auto", cmap="YlGnBu")
    ax.set_title("How competing candidates rise and fall across checkpoints")
    ax.set_xlabel("Residual checkpoint")
    ax.set_ylabel("Candidate token")
    ax.set_xticks(np.arange(len(checkpoint_labels)))
    ax.set_xticklabels(checkpoint_labels, rotation=90)
    ax.set_yticks(np.arange(len(token_labels)))
    ax.set_yticklabels(token_labels)
    fig.colorbar(image, ax=ax, label="Probability at final position")
    return fig


def plot_residual_norm(checkpoint_df):
    x = checkpoint_df["checkpoint_idx"]
    fig, ax = plt.subplots(figsize=(16, 4), constrained_layout=True)
    ax.plot(x, checkpoint_df["residual_norm"], marker="o", color=COLORS["neutral"], linewidth=2)
    ax.set_title("Residual stream norm keeps changing across the same checkpoints")
    ax.set_xlabel("Residual checkpoint")
    ax.set_ylabel("Residual L2 norm")
    ax.set_xticks(x)
    ax.set_xticklabels(checkpoint_df["label_short"], rotation=90)
    return fig


In [ ]:
def plot_component_attribution(dla_df, target_answer, top_n=12):
    plot_df = dla_df.head(top_n).sort_values("logit_attr")
    colors = [COLORS["target"] if value >= 0 else COLORS["rival"] for value in plot_df["logit_attr"]]
    fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)
    ax.barh(plot_df["component"], plot_df["logit_attr"], color=colors)
    ax.axvline(0, color=COLORS["neutral"], linewidth=1)
    ax.set_title(f"Which components push the logit for {visible_token(target_answer)}?")
    ax.set_xlabel("Direct logit attribution at the final position")
    ax.set_ylabel("Residual component")
    return fig


def plot_attention_heads(cache, head_df, str_tokens):
    heads = list(head_df[["layer", "head", "logit_attr"]].itertuples(index=False, name=None))
    fig, axes = plt.subplots(1, len(heads), figsize=(5 * len(heads), 4.5), constrained_layout=True)
    if len(heads) == 1:
        axes = [axes]
    tick_labels = [visible_token(token) for token in str_tokens]
    for ax, (layer, head, score) in zip(axes, heads):
        pattern = cache["pattern", layer][head].detach().cpu().numpy()
        image = ax.imshow(pattern, cmap="Blues")
        ax.set_title(f"L{layer}H{head}\nhead DLA={score:0.3f}")
        ax.set_xticks(np.arange(len(tick_labels)))
        ax.set_xticklabels(tick_labels, rotation=45, ha="right")
        ax.set_yticks(np.arange(len(tick_labels)))
        ax.set_yticklabels(tick_labels)
        ax.set_xlabel("Source position")
        ax.set_ylabel("Destination position")
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    return fig


def plot_block_patching(block_scores, token_labels):
    patch_labels = ["resid_pre", "attn_out", "mlp_out"]
    vmin = min(float(block_scores.min().item()), -0.25)
    vmax = max(float(block_scores.max().item()), 1.0)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), constrained_layout=True)
    for axis, patch_label, values in zip(axes, patch_labels, block_scores):
        image = axis.imshow(values.detach().cpu().numpy(), aspect="auto", cmap="RdYlBu_r", norm=norm)
        axis.set_title(patch_label)
        axis.set_xlabel("Position")
        axis.set_ylabel("Layer")
        axis.set_xticks(np.arange(len(token_labels)))
        axis.set_xticklabels(token_labels, rotation=45, ha="right")
        axis.set_yticks(np.arange(values.shape[0]))
        fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    fig.suptitle("How much clean information is restored when we patch each block output?")
    return fig


def plot_head_patching(head_scores):
    vmin = min(float(head_scores.min().item()), -0.25)
    vmax = max(float(head_scores.max().item()), 1.0)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
    image = ax.imshow(head_scores.detach().cpu().numpy(), aspect="auto", cmap="RdYlBu_r", norm=norm)
    ax.set_title("Which attention heads restore the clean answer when we patch their outputs?")
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    ax.set_xticks(np.arange(head_scores.shape[1]))
    ax.set_yticks(np.arange(head_scores.shape[0]))
    fig.colorbar(image, ax=ax, label="Normalized recovery")
    return fig


def plot_ablation_effects(ablation_df):
    plot_df = ablation_df.sort_values("delta")
    colors = [COLORS["rival"] if value < 0 else COLORS["accent"] for value in plot_df["delta"]]
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    ax.barh(plot_df["label"], plot_df["delta"], color=colors)
    ax.axvline(0, color=COLORS["neutral"], linewidth=1)
    ax.set_title("How much does the target-vs-rival score move when we ablate a candidate head?")
    ax.set_xlabel("Delta after ablation")
    ax.set_ylabel("Head")
    return fig


## 1. Start with a single question

We first anchor the investigation around one short in-context capital probe. For `gpt2-small`, that extra example sentence gives us a much cleaner answer to explain than the bare prompt alone. Starting from one concrete behavior keeps the later mechanistic claims grounded.


In [ ]:
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.set_use_attn_result(True)

tokens = model.to_tokens(PROMPT)
str_tokens = model.to_str_tokens(PROMPT)

target_token_id = safe_to_single_token(model, TARGET_ANSWER)
rival_token_id = safe_to_single_token(model, RIVAL_ANSWER)

experiment_df = pd.DataFrame(
    [
        {"field": "model", "value": MODEL_NAME},
        {"field": "prompt", "value": PROMPT},
        {"field": "string tokens", "value": str(str_tokens)},
        {"field": "target token", "value": f"{visible_token(TARGET_ANSWER)} -> {target_token_id}"},
        {"field": "rival token", "value": f"{visible_token(RIVAL_ANSWER)} -> {rival_token_id}"},
    ]
)
display(experiment_df)


In [ ]:
with torch.no_grad():
    logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)

final_logits = final_token_logits(logits).squeeze(0)
final_predictions_df = top_predictions_table(model, final_logits, k=TOP_K)
display(final_predictions_df)

target_rank = get_token_rank(final_logits, target_token_id)
rival_rank = get_token_rank(final_logits, rival_token_id)
print(f"Final rank of {visible_token(TARGET_ANSWER)}: {target_rank}")
print(f"Final rank of {visible_token(RIVAL_ANSWER)}: {rival_rank}")


In [ ]:
display_takeaways(
    title="Why start with the final prediction?",
    what="The model's next-token distribution after the full forward pass.",
    change=(f"For this prompt, {visible_token(TARGET_ANSWER)} finishes at rank {target_rank}, while {visible_token(RIVAL_ANSWER)} finishes at rank {rival_rank}."),
    why="Before we decompose anything, we want to verify that the model actually gives us a clean answer to explain.",
    practice="A mechanistic story is only useful if it is tied to a concrete behavior that would matter in a real application or regression test.",
)


## 2. Watch the answer form checkpoint by checkpoint

The next question is whether the answer appears suddenly at the end or accumulates gradually. We inspect the residual stream at every checkpoint, including pre-attention, post-attention, and post-MLP states.


In [ ]:
accum_resid, checkpoint_labels = cache.accumulated_resid(layer=-1, incl_mid=True, return_labels=True)

resid_last_pos = accum_resid[:, -1, :]
resid_normed = model.ln_final(accum_resid)
checkpoint_logits = model.unembed(resid_normed)
checkpoint_logits_last_pos = checkpoint_logits[:, -1, :]

checkpoint_df = build_checkpoint_dataframe(
    model=model,
    checkpoint_labels=checkpoint_labels,
    checkpoint_logits_last_pos=checkpoint_logits_last_pos,
    resid_last_pos=resid_last_pos,
    target_token=target_token_id,
    rival_token=rival_token_id,
)

candidate_token_ids = choose_candidate_token_ids(checkpoint_logits_last_pos, target_token=target_token_id, rival_token=rival_token_id, max_tokens=7)
candidate_labels, candidate_matrix = build_candidate_matrix(model, checkpoint_logits_last_pos, candidate_token_ids)

display(checkpoint_df.head(12))


In [ ]:
plot_emergence_overview(checkpoint_df)
plt.show()


In [ ]:
target_beats_rival = checkpoint_df.loc[checkpoint_df["logit_diff"] > 0, "label_short"]
target_reaches_top = checkpoint_df.loc[checkpoint_df["target_rank"] == 1, "label_short"]
first_target_beats_rival = target_beats_rival.iloc[0] if not target_beats_rival.empty else "never"
first_target_top = target_reaches_top.iloc[0] if not target_reaches_top.empty else "never"
opening_top = checkpoint_df.iloc[0]["top_prediction"]

display_takeaways(
    title="Reading the emergence plot",
    what="The target probability, rival probability, and target-minus-rival logit gap at each residual checkpoint.",
    change=(f"The checkpoint sequence opens with {opening_top} as the top candidate, {visible_token(TARGET_ANSWER)} first beats {visible_token(RIVAL_ANSWER)} at {first_target_beats_rival}, and it first becomes rank 1 at {first_target_top}."),
    why="This shows that the answer is assembled over time rather than appearing in one final jump.",
    practice="When a model flips from a distractor to the right answer, this timeline tells us where to inspect for the decisive internal change.",
)


In [ ]:
plot_candidate_heatmap(candidate_labels, candidate_matrix, checkpoint_df["label_short"].tolist())
plt.show()


In [ ]:
strongest_competitor_idx = int(np.argmax(candidate_matrix[1:].max(axis=1))) + 1 if candidate_matrix.shape[0] > 1 else 0
strongest_competitor = candidate_labels[strongest_competitor_idx]
strongest_competitor_peak = candidate_matrix[strongest_competitor_idx].max()
target_peak = candidate_matrix[0].max()

display_takeaways(
    title="Reading the competition heatmap",
    what="A compact view of how a small set of plausible tokens rise and fall over the same checkpoints.",
    change=(f"The strongest non-target candidate in this set is {strongest_competitor}, which peaks at probability {strongest_competitor_peak:0.3f}, while {candidate_labels[0]} eventually peaks at {target_peak:0.3f}."),
    why="A good mechanistic story should account for competing answers, not just the final winner.",
    practice="This is the kind of view that helps teams distinguish a clean correction from a brittle answer that only wins at the last moment.",
)


In [ ]:
plot_residual_norm(checkpoint_df)
plt.show()


In [ ]:
biggest_norm_jump = checkpoint_df["residual_norm"].diff().abs().fillna(0).idxmax()
biggest_norm_label = checkpoint_df.loc[biggest_norm_jump, "label_short"]
biggest_norm_delta = checkpoint_df["residual_norm"].diff().abs().fillna(0).max()

display_takeaways(
    title="Reading the residual norm plot",
    what="The L2 norm of the accumulated residual stream at the final position across the same checkpoints.",
    change=(f"The largest checkpoint-to-checkpoint norm change lands at {biggest_norm_label}, with an absolute jump of {biggest_norm_delta:0.3f}."),
    why="Confidence shifts and residual magnitude shifts are related, but they are not identical. The answer can improve while the overall state is still being substantially rewritten.",
    practice="If a deployment regression only shows up at a few checkpoints, norm changes can help narrow the search to a smaller part of the forward pass.",
)


## 3. Which components push the answer upward?

Descriptive plots tell us when the answer becomes visible. Direct logit attribution gives us the next layer of the story: which residual components are pushing the target token at the final position.


In [ ]:
resid_all, labels_all = cache.decompose_resid(layer=model.cfg.n_layers, mode="all", return_labels=True)
resid_attn, labels_attn = cache.decompose_resid(layer=model.cfg.n_layers, mode="attn", return_labels=True)
resid_mlp, labels_mlp = cache.decompose_resid(layer=model.cfg.n_layers, mode="mlp", return_labels=True)
head_results = cache.stack_head_results(layer=model.cfg.n_layers, return_labels=False)

dla_all_df = attrs_to_df(labels_all, cache.logit_attrs(resid_all, TARGET_ANSWER))
dla_attn_df = attrs_to_df(labels_attn, cache.logit_attrs(resid_attn, TARGET_ANSWER))
dla_mlp_df = attrs_to_df(labels_mlp, cache.logit_attrs(resid_mlp, TARGET_ANSWER))
head_attr_df = head_attrs_to_df(model, cache.logit_attrs(head_results, TARGET_ANSWER))

display(Markdown("### Top residual contributors"))
display(dla_all_df.head(10))
display(Markdown("### Top attention heads by direct logit attribution"))
display(head_attr_df.head(10))


In [ ]:
plot_component_attribution(dla_all_df, TARGET_ANSWER, top_n=TOP_COMPONENTS)
plt.show()


In [ ]:
top_component = dla_all_df.iloc[0]
top_attention_head = head_attr_df.iloc[0]

display_takeaways(
    title="Reading the attribution view",
    what="Direct logit attribution scores for residual components and individual attention heads at the final position.",
    change=(f"The strongest overall component is {top_component['component']} with attribution {top_component['logit_attr']:0.3f}, and the strongest attention head is {top_attention_head['label']} with attribution {top_attention_head['logit_attr']:0.3f}."),
    why="This is our first concrete hypothesis about where the answer is being pushed upward inside the network.",
    practice="Attribution narrows a large model into a short list of components worth auditing, testing, or monitoring after model changes.",
)


## 4. Do the candidate heads attend to the right places?

Attention patterns are not causal proof, but they are useful supporting evidence. Once attribution points us toward a few heads, we can inspect whether their routing behavior looks consistent with the story we want to test.


In [ ]:
attention_heads_df = head_attr_df.head(HEADS_TO_VISUALIZE).copy()
plot_attention_heads(cache, attention_heads_df, str_tokens)
plt.show()


In [ ]:
top_head_row = attention_heads_df.iloc[0]
top_pattern = cache["pattern", int(top_head_row["layer"])][int(top_head_row["head"])].detach().cpu().numpy()
top_last_query = top_pattern[-1]
top_source_idx = int(top_last_query.argmax())
top_source_token = visible_token(str_tokens[top_source_idx])
top_source_weight = float(top_last_query[top_source_idx])

display_takeaways(
    title="Reading the attention patterns",
    what="Static attention maps for the most attribution-heavy heads.",
    change=(f"For the top head {top_head_row['label']}, the final query position pays its largest attention weight to {top_source_token} with weight {top_source_weight:0.3f}."),
    why="When a candidate head attends to a token that plausibly supports the answer, the mechanistic story becomes more coherent before we even intervene causally.",
    practice="Qualitative routing checks help us spot whether a high-scoring head looks interpretable or whether it may just be a spurious correlate worth testing carefully.",
)


## 5. Can we move from description to causality?

Now we compare a clean prompt with a corrupted prompt and patch activations from the clean run into the corrupted run. This lets us ask which blocks and which heads actually restore the right answer rather than merely correlating with it.


In [ ]:
clean_tokens = model.to_tokens(CLEAN_PROMPT)
corrupted_tokens = model.to_tokens(CORRUPTED_PROMPT)
assert clean_tokens.shape == corrupted_tokens.shape, "Clean and corrupted prompts must have the same token length."

with torch.inference_mode():
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

metric = make_metric_for_answer(target_token_id, rival_token_id)
clean_score = metric(clean_logits).item()
corrupted_score = metric(corrupted_logits).item()

patch_setup_df = pd.DataFrame(
    [
        {"condition": "clean", "prompt": CLEAN_PROMPT, "metric": clean_score},
        {"condition": "corrupted", "prompt": CORRUPTED_PROMPT, "metric": corrupted_score},
    ]
)
display(patch_setup_df)


In [ ]:
block_patch = patching.get_act_patch_block_every(model=model, corrupted_tokens=corrupted_tokens, clean_cache=clean_cache, metric=metric).detach().cpu()
block_patch_normalized = normalize_patch_scores(block_patch, clean_score=clean_score, corrupted_score=corrupted_score)
plot_block_patching(block_patch_normalized, [visible_token(token) for token in model.to_str_tokens(CLEAN_PROMPT)])
plt.show()


In [ ]:
patch_labels = ["resid_pre", "attn_out", "mlp_out"]
best_block_flat_idx = int(block_patch_normalized.argmax().item())
best_patch_type_idx, best_layer_idx, best_position_idx = np.unravel_index(best_block_flat_idx, block_patch_normalized.shape)
best_patch_type = patch_labels[best_patch_type_idx]
best_patch_token = visible_token(model.to_str_tokens(CLEAN_PROMPT)[best_position_idx])
best_patch_score = block_patch_normalized[best_patch_type_idx, best_layer_idx, best_position_idx].item()

display_takeaways(
    title="Reading the block patching heatmaps",
    what="Normalized recovery scores showing how much of the clean target-vs-rival behavior comes back when we patch each block output into the corrupted run.",
    change=(f"The strongest block-level recovery appears at {best_patch_type} in layer {best_layer_idx} at token position {best_patch_token}, with normalized recovery {best_patch_score:0.3f}."),
    why="This tells us where the clean evidence for the answer is actually stored in a causal sense, not just where the final logits happen to look favorable.",
    practice="Patching is especially useful for debugging when we want to know which layer and component need attention after a behavior changes between two closely related prompts.",
)


In [ ]:
head_patch_all = patching.get_act_patch_attn_head_all_pos_every(model=model, corrupted_tokens=corrupted_tokens, clean_cache=clean_cache, metric=metric).detach().cpu()
head_out_patch_normalized = normalize_patch_scores(head_patch_all[0], clean_score=clean_score, corrupted_score=corrupted_score)
head_patch_df = head_scores_to_df(head_out_patch_normalized)

display(head_patch_df.head(12))
plot_head_patching(head_out_patch_normalized)
plt.show()


In [ ]:
top_patch_head = head_patch_df.iloc[0]

display_takeaways(
    title="Reading the head patching heatmap",
    what="Normalized recovery after patching each attention head's output into the corrupted run.",
    change=(f"The strongest head-level recovery comes from {top_patch_head['label']} with normalized recovery {top_patch_head['score']:0.3f}."),
    why="If a small number of heads restore most of the clean behavior, we have a much tighter mechanistic hypothesis than 'the whole model did it.'",
    practice="This gives us a shortlist of candidate heads for targeted audits, ablations, or regression checks after a model update.",
)


## 6. What breaks if we remove the candidate heads?

Patching tells us which heads can restore the clean behavior. Ablation asks the complementary question: if we remove those heads on the clean prompt, how much of the answer falls apart?


In [ ]:
candidate_heads_df = head_patch_df.head(HEADS_TO_ABLATE).copy()
with torch.inference_mode():
    baseline_metric = metric(model(clean_tokens)).item()

ablation_rows = []
for row in candidate_heads_df.itertuples(index=False):
    ablation_rows.append(
        ablate_single_head_and_score(
            model=model,
            tokens=clean_tokens,
            layer=int(row.layer),
            head=int(row.head),
            metric_fn=metric,
            baseline_score=baseline_metric,
        )
    )

ablation_df = pd.DataFrame(ablation_rows).sort_values("delta").reset_index(drop=True)
display(ablation_df)


In [ ]:
plot_ablation_effects(ablation_df)
plt.show()


In [ ]:
most_harmful_head = ablation_df.iloc[0]

display_takeaways(
    title="Reading the ablation chart",
    what="The change in the target-minus-rival score after zeroing out each candidate head on the clean prompt.",
    change=(f"Ablating {most_harmful_head['label']} produces the largest drop, with delta {most_harmful_head['delta']:0.3f}."),
    why="This is the strongest evidence in the notebook that a small set of heads are not just correlated with the answer. They matter to producing it.",
    practice="Ablation gives us a practical way to distinguish explanatory components from components that are merely along for the ride.",
)


## 7. What did we learn, and why does it matter beyond this prompt?

We can now compress the investigation into a short story. The answer does not appear all at once. Competing candidates rise and fall across checkpoints. Attribution and attention inspection help us form a hypothesis about the important components. Patching and ablation then tell us which parts of that hypothesis survive causal testing.


In [ ]:
final_summary_lines = [
    "### Final summary",
    f"- The target first beats the rival at: {first_target_beats_rival}",
    f"- The target first reaches rank 1 at: {first_target_top}",
    f"- Strongest overall residual contributor: {top_component['component']} ({top_component['logit_attr']:0.3f})",
    f"- Strongest attention head by attribution: {top_attention_head['label']} ({top_attention_head['logit_attr']:0.3f})",
    f"- Strongest block patch site: {best_patch_type} in layer {best_layer_idx} on {best_patch_token} ({best_patch_score:0.3f})",
    f"- Strongest head patch site: {top_patch_head['label']} ({top_patch_head['score']:0.3f})",
    f"- Most harmful head ablation: {most_harmful_head['label']} ({most_harmful_head['delta']:0.3f})",
]
display(Markdown("\n".join(final_summary_lines)))


## Why this style of analysis matters in practice

- Debugging: We can localize where a good answer turns into a bad one, or where a bad answer gets corrected.
- Trust and governance: We get stronger evidence than surface-level prompting alone when we need to explain why a model behaves the way it does.
- Client-facing model understanding: We can move from vague claims about "the model knows this" to a concrete story about which internal computations supported the answer.
- Change management: After fine-tuning, distillation, or model upgrades, these tools let us check whether the same behavior is being implemented by the same internal machinery.

This notebook is still a local story rather than a universal theorem. That is part of the point. Mechanistic understanding improves when we make a concrete hypothesis, test it on a real run, and stay honest about what the evidence does and does not show.


## Appendix: optional MLP / neuron drill-down

The main story above is already complete. If we want to keep digging, we can decompose the residual stream more finely and inspect which MLP outputs contribute to the target token.


In [ ]:
try:
    full_resid, full_labels = cache.get_full_resid_decomposition(layer=model.cfg.n_layers, return_labels=True)
    full_attrs = cache.logit_attrs(full_resid, TARGET_ANSWER)
    full_df = attrs_to_df(full_labels, full_attrs)
    display(full_df.head(25))
except Exception as exc:
    print("Optional neuron-level drill-down was skipped.")
    print("Reason:", repr(exc))
